# Academic Admission Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `admitted`

## 2 · Project Overview

We predict whether a graduate school applicant will be **admitted** based on GRE score, GPA, research experience, personal statement, recommendations, and university ranking.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given an applicant's GRE score, GPA, research experience, statement score, recommendation score, and target university rank, predict admission outcome.

## 5 · Why This Project Matters

- **Admission prediction** helps applicants target realistic schools.
- Universities can benchmark their admission criteria.
- Understanding which factors matter most aids preparation.
- Classic ML classification problem in education analytics.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 5,000 |
| **Features** | gre_score, gpa, research_exp, statement_score, recommendation_score, university_rank |
| **Target** | admitted (0 = rejected, 1 = admitted) |
| **Class balance** | ~50/50 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "admitted"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: admitted
Save dir: E:\Github\Machine-Learning-Projects\Classification\Academic Admission Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 5,000 applicants with academic and demographic features.

In [4]:
np.random.seed(SEED)
n = 5000
gre_score = np.random.randint(260, 340, n)
gpa = np.round(np.random.uniform(2.0, 4.0, n), 2)
research_exp = np.random.binomial(1, 0.45, n)
statement_score = np.round(np.random.uniform(1, 5, n), 1)
recommendation_score = np.round(np.random.uniform(1, 5, n), 1)
university_rank = np.random.choice([1, 2, 3, 4], n, p=[0.15, 0.3, 0.35, 0.2])

score = (0.01 * (gre_score - 280) + 0.5 * (gpa - 2.5) + 0.3 * research_exp
         + 0.15 * statement_score + 0.1 * recommendation_score - 0.15 * university_rank
         + np.random.normal(0, 0.6, n))
admitted = (score > np.median(score)).astype(int)

df = pd.DataFrame({"gre_score": gre_score, "gpa": gpa, "research_exp": research_exp,
                    "statement_score": statement_score, "recommendation_score": recommendation_score,
                    "university_rank": university_rank, "admitted": admitted})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['admitted'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (5000, 7)
Class balance:
admitted
1    0.5
0    0.5
Name: proportion, dtype: float64


,gre_score,gpa,research_exp,statement_score,recommendation_score,university_rank,admitted
0,311,2.34,0,4.2,1.3,2,1
1,274,3.65,1,4.9,4.5,3,0
2,331,3.17,1,4.7,1.7,3,1
3,320,2.68,0,3.9,3.5,3,1
4,280,3.61,1,3.8,3.6,1,1


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (5000, 7)

Missing values:
gre_score               0
gpa                     0
research_exp            0
statement_score         0
recommendation_score    0
university_rank         0
admitted                0
dtype: int64

Duplicate rows: 0

Dtypes:
gre_score                 int32
gpa                     float64
research_exp              int32
statement_score         float64
recommendation_score    float64
university_rank           int64
admitted                  int64
dtype: object

Target 'admitted' confirmed. Value counts:
admitted
1    2500
0    2500
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["gre_score", "gpa", "statement_score", "recommendation_score"]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 2][i % 2]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Admission Status", fontsize=14)
plt.tight_layout()
plt.show()

corr = df.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `admitted`.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["salmon", "steelblue"], edgecolor="black")
axes[0].set_title("Admission Distribution")
axes[0].set_xticklabels(["Rejected (0)", "Admitted (1)"], rotation=0)

df.groupby("university_rank")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Admission Rate by University Rank")
axes[1].set_ylabel("Admission Rate")
plt.tight_layout()
plt.show()
print(f"Admission rate: {df[TARGET].mean():.1%}")

Admission rate: 50.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (4000, 6), Test: (1000, 6)
Train class distribution:
admitted
1    0.5
0    0.5
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: None — all numeric (university_rank is ordinal).
- **Scaling**: Not needed for tree models.
- **Class balance**: ~50/50 by design.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["academic_index"] = X_train["gre_score"] / 340 * 0.5 + X_train["gpa"] / 4.0 * 0.5
X_test["academic_index"] = X_test["gre_score"] / 340 * 0.5 + X_test["gpa"] / 4.0 * 0.5

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (7): ['gre_score', 'gpa', 'research_exp', 'statement_score', 'recommendation_score', 'university_rank', 'academic_index']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7160
F1       : 0.7171

              precision    recall  f1-score   support

           0       0.72      0.71      0.71       500
           1       0.71      0.72      0.72       500

    accuracy                           0.72      1000
   macro avg       0.72      0.72      0.72      1000
weighted avg       0.72      0.72      0.72      1000



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                     
SVC                            0.720              0.720  0.787640  0.719982   0.720056   0.720    0.548600
RidgeClassifierCV              0.720              0.720  0.804836  0.719999   0.720004   0.720    0.018215
LinearDiscriminantAnalysis     0.719              0.719  0.804892  0.718997   0.719008   0.719    0.018515
RidgeClassifier                0.719              0.719  0.804896  0.718997   0.719008   0.719    0.018937
AdaBoostClassifier             0.718              0.718  0.792272  0.717959   0.718126   0.718    0.149334
CalibratedClassifierCV         0.718              0.718  0.805032  0.717990   0.718031   0.718    0.045167
LinearSVC                      0.718              0.718  0.805004  0.717990   0.718031   0.718    0.016554
Log

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.7230
F1: 0.7210


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.7186  (1.0s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[59]	valid_0's binary_logloss: 0.557478
LightGBM F1: 0.7181  (0.8s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                   0.723  0.7210     0.7262   0.716
CatBoost                0.718  0.7186     0.7171   0.720
LightGBM                0.715  0.7181     0.7104   0.726
Logistic Regression     0.716  0.7171     0.7143   0.720

Top 3 models: ['FLAML', 'CatBoost', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.7230
    F1       : 0.7210
    Precision: 0.7262
    Recall   : 0.7160

  CatBoost:
    Accuracy : 0.7180
    F1       : 0.7186
    Precision: 0.7171
    Recall   : 0.7200

  LightGBM:
    Accuracy : 0.7150
    F1       : 0.7181
    Precision: 0.7104
    Recall   : 0.7260


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.73      0.72       500
           1       0.73      0.72      0.72       500

    accuracy                           0.72      1000
   macro avg       0.72      0.72      0.72      1000
weighted avg       0.72      0.72      0.72      1000


Total errors: 277 / 1000 (27.7%)

Sample misclassifications:
      gre_score   gpa  research_exp  statement_score  recommendation_score  university_rank  academic_index  actual  predicted  correct
2923        300  3.02             1              2.1                   2.4                2        0.818676       0          1    False
2645        270  2.86             1              3.4                   2.5                3        0.754559       1          0    False
2387        329  3.12             0              4.9                   1.1                3        0.873824       0          1    False
1731        339  2.86    

## 25 · Interpretation and Business Insight

**Key findings:**
- **GPA** and **GRE score** are the strongest admission predictors.
- **Research experience** provides a significant boost.
- **University rank** matters — higher-ranked schools are harder to enter.
- **Statement and recommendation** scores have moderate effects.

**Business takeaway:** Focus on GPA and GRE, but research experience is a strong differentiator.

## 26 · Limitations

1. Synthetic data with simplified admission criteria.
2. No extracurricular activities or work experience.
3. Missing field-of-study specifics.
4. No interview or portfolio evaluation.
5. University rank is a crude proxy for selectivity.

## 27 · How to Improve This Project

1. Use real admission data (with privacy protection).
2. Add field-specific requirements.
3. Include work experience and extracurriculars.
4. Model probability of admission as a score.
5. Add country/residency status as a factor.

## 28 · Production Considerations

- Deploy as an admission chance calculator tool.
- Output probability ranges, not binary predictions.
- Include disclaimer about model limitations.
- Update annually with new admission cycles.
- Avoid reinforcing bias — audit for fairness.

## 29 · Common Mistakes

1. Treating admission as purely quantitative.
2. Not considering university-specific criteria.
3. Ignoring fairness and bias in predictions.
4. Over-relying on GRE (some programs are GRE-optional).
5. Not validating across different admission cycles.

## 30 · Mini Challenge / Exercises

1. Remove `gre_score` — how much does F1 change?
2. Interaction: `gpa * research_exp` — does it help?
3. Separate analysis for top-rank vs. lower-rank universities.
4. Try probability calibration and plot reliability diagram.
5. Examine if the model is biased toward any feature range.

## 31 · Final Summary / Key Takeaways

1. **GPA and GRE** are the dominant admission factors.
2. **Research experience** is a strong differentiator.
3. **University rank** creates different admission thresholds.
4. **Tree-based models** capture non-linear interactions.
5. **Fairness auditing** is critical in educational ML.
6. **Probability estimates** are more useful than binary predictions.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Academic Admission Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.718,
    "f1": 0.7186,
    "precision": 0.7171,
    "recall": 0.72
  },
  "LightGBM": {
    "accuracy": 0.715,
    "f1": 0.7181,
    "precision": 0.7104,
    "recall": 0.726
  },
  "Logistic Regression": {
    "accuracy": 0.716,
    "f1": 0.7171,
    "precision": 0.7143,
    "recall": 0.72
  },
  "FLAML": {
    "accuracy": 0.723,
    "f1": 0.721,
    "precision": 0.7262,
    "recall": 0.716
  }
}
